# AutoShop mit Responses API Multi-agent

Dieses Notebook zeigt die **Multi-agent-Funktion direkt in der OpenAI Responses API**.

Im Gegensatz zu den anderen Varianten werden die Subagents nicht als Python-Objekte
mit dem Agents SDK definiert und auch nicht über einen selbstgebauten Graph gesteuert.

Stattdessen wird Multi-agent direkt im Responses-Request aktiviert:

```python
multi_agent={
    "enabled": True,
    "max_concurrent_subagents": 3,
}
```

Die Responses API stellt dem Root-Agent intern Kollaborationsprimitive wie
`spawn_agent`, `send_message`, `followup_task` und `wait_agent` zur Verfügung.

```text
                          User
                           |
                           v
                         /root
                           |
            +--------------+--------------+
            |              |              |
            v              v              v
   /root/autoshop  /root/research  /root/offer
            |              |              |
            +--------------+--------------+
                           |
                    gemeinsame Tools
                           |
                           v
                  AutoShop MCP Bridge
```

**Pattern:** Responses API Multi-agent / Dynamic Agent Team

Wichtig: Multi-agent ist eine Beta-Funktion der OpenAI Responses API.


---

## Unterschiede zu den bisherigen Varianten

| Variante | Orchestrierung |
|---|---|
| Responses API klassisch | Anwendung führt Tool-Loop aus |
| Agents SDK | Python definiert explizite `Agent(...)`-Objekte |
| Agents SDK Orchestrator | Manager verwendet `Agent.as_tool()` |
| Python Graph | Python-State-Graph bestimmt Nodes und Routing |
| **Responses API Multi-agent** | **Responses API erzeugt und koordiniert Subagents dynamisch** |

Bei Responses Multi-agent teilen Root-Agent und Subagents dasselbe Modell und dieselben
im Request konfigurierten Tools. Die fachliche Rollentrennung wird deshalb im Prompt
vorgegeben.


---

## Voraussetzungen

Dieses Notebook benötigt:

- eine aktuelle OpenAI Python SDK-Version mit Beta Responses Multi-agent,
- ein OpenAI-Modell, das Responses Multi-agent unterstützt,
- den laufenden AutoShop MCP-Server.

Die Beta wird pro Request mit

```python
betas=["responses_multi_agent=v1"]
```

aktiviert.


In [ ]:
%pip install -q -U openai "mcp>=1.19,<3"

---

## Umgebung und OpenAI Client

Das bestehende `~/data/env.py` wird weiterverwendet.

Für Responses Multi-agent muss `AI_BASE_URL` auf einen Endpoint zeigen, der diese
OpenAI-Beta unterstützt. Ein beliebiger OpenAI-kompatibler Endpoint implementiert
diese Funktion nicht automatisch.


In [ ]:
%run ~/data/env.py
AI_MODEL = "gpt-5.6-sol"

import json
import subprocess
from collections import defaultdict

from openai import AsyncOpenAI

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True,
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True,
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"


client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

MCP_URL = get_server_url()

print(f"OpenAI Base URL: {AI_BASE_URL}")
print(f"Model:           {AI_MODEL}")
print(f"MCP URL:         {MCP_URL}")


---

# 1. MCP-Tools als Responses Function Tools

Der AutoShop MCP-Server läuft lokal im Kubernetes-Cluster.

Da die Responses API diesen lokalen NodePort nicht direkt erreichen kann, werden die
MCP-Tools wie im ursprünglichen Notebook als **developer-defined Function Tools**
an die Responses API exponiert.

Wenn Root-Agent oder ein Subagent ein solches Tool aufruft:

```text
Responses Multi-agent
        |
        v
function_call
        |
        v
dieses Notebook
        |
        v
MCP ClientSession
        |
        v
AutoShop MCP Server
```

Die Multi-Agent-Kollaboration selbst wird dagegen von der Responses API ausgeführt.


In [ ]:
READ_ONLY_TOOLS = {
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
}


def mcp_tool_to_openai_tool(tool):
    schema = tool.input_schema or {
        "type": "object",
        "properties": {},
    }

    # Responses Function Tool
    return {
        "type": "function",
        "name": tool.name,
        "description": (
            getattr(tool, "description", None)
            or f"AutoShop MCP Tool {tool.name}"
        ),
        "parameters": schema,
    }


def mcp_result_to_text(result):
    parts = []

    for block in result.content:
        if hasattr(block, "text"):
            parts.append(block.text)
        elif hasattr(block, "model_dump"):
            parts.append(
                json.dumps(
                    block.model_dump(),
                    ensure_ascii=False,
                )
            )
        else:
            parts.append(str(block))

    return "\n".join(parts)


---

# 2. Root-Agent Prompt

Es werden **keine festen Python-Agenten definiert**.

Der Root-Agent erhält stattdessen die Aufgabe, bei Bedarf selbst Subagents zu erzeugen.

Für das Demo-Szenario sollen drei Rollen verwendet werden:

- `autoshop` – interne AutoShop-Daten über MCP
- `research` – allgemeine Zusatzinformationen
- `offer` – strukturierte Offerte

Da alle Subagents dasselbe Toolset sehen, wird zusätzlich festgelegt, welche Rolle
welche Tools fachlich verwenden darf.


In [ ]:
ROOT_INSTRUCTIONS = """
Du bist /root und leitest ein dynamisches AutoShop-Agententeam.

Ziel:
Bearbeite den Kundenwunsch vollständig und liefere am Ende eine konsistente
deutsche Antwort oder Markdown-Offerte.

Delegation:
- Erzeuge bei Bedarf einen Subagent mit der Rolle "autoshop".
  Dieser Agent soll interne AutoShop-Daten über die bereitgestellten
  Catalog-, Customer- und Order-Tools beschaffen.
- Erzeuge bei Bedarf einen Subagent mit der Rolle "research".
  Dieser Agent soll nur allgemeine Zusatzinformationen, typische Vorteile,
  Kaufkriterien und unverbindliche Extras liefern.
- Erzeuge bei Bedarf einen Subagent mit der Rolle "offer".
  Dieser Agent soll aus bereits bekannten Fakten eine strukturierte
  Markdown-Offerte erstellen.

Regeln:
- Shop-Fahrzeuge, Preise, IDs, Kunden- und Auftragsdaten dürfen niemals erfunden werden.
- Für konkrete AutoShop-Daten müssen die bereitgestellten Function Tools verwendet werden.
- Der Research-Agent soll keine internen AutoShop-Daten beschaffen oder erfinden.
- Der Offer-Agent darf keine neuen Shop-Fakten oder Preise erfinden.
- Wenn unabhängige Teilaufgaben parallel ausgeführt werden können, delegiere sie parallel.
- Führe die Resultate der Subagents am Ende selbst zusammen.
- Die finale Antwort muss vom Root-Agent kommen.
- Antworte auf Deutsch.
"""


---

# 3. Finalen Root-Text extrahieren

Bei Multi-agent enthalten Output-Items Informationen darüber, von welchem Agenten sie
stammen. Die finale Benutzerantwort wird aus der `final_answer`-Message von `/root`
extrahiert.


In [ ]:
ROOT = "/root"


def get_agent_name(item):
    agent = getattr(item, "agent", None)
    return getattr(agent, "agent_name", None) or ROOT


def extract_root_final_text(items):
    parts = []

    for item in items:
        if getattr(item, "type", None) != "message":
            continue

        if get_agent_name(item) != ROOT:
            continue

        if getattr(item, "phase", None) != "final_answer":
            continue

        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "output_text":
                parts.append(part.text)

    return "".join(parts)


---

# 4. Responses Multi-agent + MCP Tool Loop

Die Responses API übernimmt:

```text
spawn_agent
send_message
followup_task
wait_agent
Subagent-Baum
Root-Synthese
```

Das Notebook übernimmt nur noch die **developer-defined Function Calls** für den
lokalen AutoShop-MCP-Server.

Wichtig: Ein Function Call kann sowohl vom Root-Agent als auch von einem Subagent
kommen.


In [ ]:
async def run_responses_multi_agent(
    customer_name: str,
    user_request: str,
    max_rounds: int = 10,
    max_concurrent_subagents: int = 3,
    show_subagents: bool = True,
):
    async with streamable_http_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()

            mcp_tools = [
                tool
                for tool in tools_result.tools
                if tool.name in READ_ONLY_TOOLS
            ]

            tools = [
                mcp_tool_to_openai_tool(tool)
                for tool in mcp_tools
            ]

            history = [
                {
                    "role": "developer",
                    "content": ROOT_INSTRUCTIONS,
                },
                {
                    "role": "user",
                    "content": f"""
Kunde:
{customer_name}

Kundenanfrage:
{user_request}
""",
                },
            ]

            all_output_items = []
            agent_events = defaultdict(list)

            for round_no in range(1, max_rounds + 1):
                output_items = []
                pending_calls = []
                item_agents = {}

                print(f"\n--- Responses Multi-agent Runde {round_no} ---")

                stream = await client.beta.responses.create(
                    model=AI_MODEL,
                    input=history,
                    tools=tools,
                    store=False,
                    multi_agent={
                        "enabled": True,
                        "max_concurrent_subagents": max_concurrent_subagents,
                    },
                    stream=True,
                    betas=["responses_multi_agent=v1"],
                )

                async for event in stream:
                    if event.type == "response.output_item.added":
                        agent = get_agent_name(event.item)
                        item_agents[event.output_index] = agent

                    elif event.type == "response.output_text.delta":
                        agent = item_agents.get(event.output_index, ROOT)

                        if agent == ROOT:
                            print(event.delta, end="", flush=True)
                        elif show_subagents:
                            print(
                                f"[{agent}] {event.delta}",
                                end="",
                                flush=True,
                            )

                    elif event.type == "response.output_item.done":
                        item = event.item
                        output_items.append(item)

                        agent = get_agent_name(item)
                        agent_events[agent].append(item)

                        if item.type == "function_call":
                            pending_calls.append(item)

                    elif event.type == "response.completed":
                        usage = getattr(event.response, "usage", None)
                        if usage is not None:
                            print(f"\nUsage: {usage}")

                    elif event.type in {
                        "error",
                        "response.failed",
                        "response.incomplete",
                    }:
                        raise RuntimeError(str(event))

                # Beta Output Items dürfen als Input in der nächsten Runde
                # wiederverwendet werden.
                history.extend(output_items)
                all_output_items.extend(output_items)

                # Developer-defined Functions ausführen.
                for call in pending_calls:
                    args = json.loads(call.arguments or "{}")

                    print(
                        f"\n[MCP] {get_agent_name(call)} -> "
                        f"{call.name}({args})"
                    )

                    tool_result = await session.call_tool(
                        call.name,
                        arguments=args,
                    )

                    history.append(
                        {
                            "type": "function_call_output",
                            "call_id": call.call_id,
                            "output": mcp_result_to_text(tool_result),
                        }
                    )

                # Wenn keine developer-defined Function Calls mehr offen sind,
                # ist der Multi-Agent-Run abgeschlossen.
                if not pending_calls:
                    break

            else:
                raise RuntimeError(
                    f"Abgebrochen: mehr als {max_rounds} Tool-Runden."
                )

            final_text = extract_root_final_text(all_output_items)

            return {
                "final_text": final_text,
                "output_items": all_output_items,
                "agent_events": dict(agent_events),
                "history": history,
            }


---

# 5. Beispiel: Fahrzeugempfehlung mit Offerte

Für diese Anfrage kann `/root` beispielsweise:

1. einen `autoshop`-Subagent für die interne Suche erzeugen,
2. parallel einen `research`-Subagent für allgemeine Zusatzinformationen erzeugen,
3. die Resultate sammeln,
4. einen `offer`-Subagent mit der Offert-Erstellung beauftragen,
5. das Endresultat als `/root` synthetisieren.

Die konkrete Agentenstruktur wird zur Laufzeit durch die Responses API bestimmt.


In [ ]:
from IPython.display import Markdown, display

result = await run_responses_multi_agent(
    customer_name="Max Muster",
    user_request=(
        "Ich suche ein günstiges Auto mit viel Platz für die Familie. "
        "Bitte empfehle mir ein passendes Fahrzeug, ergänze sinnvolle Extras "
        "und erstelle eine Demo-Offerte."
    ),
    max_concurrent_subagents=3,
)

display(Markdown(result["final_text"]))


---

# 6. Dynamisch erzeugte Agenten anzeigen

Multi-agent verwendet Agent-Namen wie:

```text
/root
/root/autoshop
/root/research
/root/offer
```

Die exakten Namen und die Tiefe des Agentenbaums werden vom Modell zur Laufzeit bestimmt.


In [ ]:
print("Im Run beobachtete Agenten:")

for agent_name, items in sorted(result["agent_events"].items()):
    print(f"- {agent_name}: {len(items)} Output-Items")


---

# 7. Output-Item-Typen pro Agent

Damit lässt sich nachvollziehen, welche Agenten Messages, Function Calls oder
Multi-agent-Kollaborationsitems erzeugt haben.


In [ ]:
for agent_name, items in sorted(result["agent_events"].items()):
    print(f"\n{agent_name}")

    for item in items:
        print(
            "  ",
            getattr(item, "type", type(item).__name__),
            getattr(item, "phase", ""),
        )


---

# 8. Multi-agent-Kollaborationsitems untersuchen

Die Responses API erzeugt eigene Multi-agent-Items für Aktionen wie das Erzeugen und
Koordinieren von Subagents.

Die genaue Item-Struktur ist Beta und kann sich ändern. Deshalb wird sie hier
explorativ ausgegeben statt fest verdrahtet.


In [ ]:
for item in result["output_items"]:
    item_type = getattr(item, "type", "")

    if "multi_agent" in item_type:
        print("=" * 100)
        print(f"Agent: {get_agent_name(item)}")
        print(f"Type:  {item_type}")
        print(item)


---

# 9. Architekturvergleich

## Agents SDK Orchestrator

```text
Python
  |
  +-- Agent("Manager")
        |
        +-- autoshop_agent.as_tool()
        +-- research_agent.as_tool()
        +-- offer_agent.as_tool()
```

Die Worker werden in Python explizit definiert.

## Responses API Multi-agent

```text
Responses API
       |
       v
     /root
       |
       +-- spawn_agent(...)
       +-- spawn_agent(...)
       +-- wait_agent(...)
       |
       v
 dynamischer Agentenbaum
```

Die Agenten werden zur Laufzeit durch die Responses API erzeugt.

## Graph-Version

```text
Python State Graph
       |
       +-- Nodes
       +-- Edges
       +-- State
       +-- Conditions
```

Hier bestimmt Python den Kontrollfluss explizit.


---

# 10. Wichtige Eigenschaft: gemeinsames Modell und gemeinsame Tools

Bei Responses Multi-agent teilen alle Agenten im Baum:

- dasselbe Modell,
- dasselbe im Request konfigurierte Toolset.

Das unterscheidet sich vom Agents SDK, wo beispielsweise nur der AutoShop-Agent
MCP-Tools besitzen kann.

In diesem Notebook wird deshalb die Rollentrennung im Root-Prompt beschrieben:

```text
autoshop -> interne MCP-Daten
research -> allgemeine Zusatzinformationen
offer    -> Offerte aus vorhandenen Fakten
```

Die technische Tool-Berechtigung ist jedoch für alle Subagents gleich.


---

# 11. HTTP versus WebSocket

Dieses Notebook verwendet **HTTP Streaming**, weil der Ablauf für ein Lehrbeispiel
leicht nachvollziehbar bleibt.

Bei developer-defined Function Calls funktioniert der Ablauf so:

```text
Agent -> function_call
          |
          v
Response beendet Tool-Phase
          |
          v
Notebook führt MCP Tool aus
          |
          v
neuer Responses Request mit function_call_output
```

Für tool-intensive oder länger laufende Multi-Agent-Workflows unterstützt OpenAI
auch WebSocket Mode mit `response.inject`. Damit kann ein Tool-Resultat direkt in
einen aktiven Multi-Agent-Run injiziert werden, ohne auf einen vollständigen
HTTP-Fortsetzungsrequest zu warten.


---

# 12. Pattern

## Responses API Multi-agent / Dynamic Agent Team

Ein Root-Agent kann zur Laufzeit eigenständig Subagents erzeugen, Aufgaben delegieren,
Nachrichten austauschen und auf Teilresultate warten. Die Responses API übernimmt
dabei die Kollaboration innerhalb des Agentenbaums und der Root-Agent synthetisiert
die finale Antwort.

```text
Root Agent
    +
dynamische Subagents
    +
Parallelisierung
    +
gemeinsame Tools
    =
Responses API Multi-agent
```

Das Pattern eignet sich besonders für Aufgaben, die sich dynamisch in mehrere
unabhängige oder teilweise parallele Workstreams zerlegen lassen.


---

# 13. Einschränkungen gegenüber einem Durable Graph

Responses Multi-agent ist **kein persistenter Workflow-Graph**.

Es bietet dynamische Agenten-Kollaboration, ersetzt aber nicht automatisch:

- persistent gespeicherten Workflow-State,
- garantierte fachliche Prozessreihenfolge,
- langlebige Wartezustände,
- explizite Retry-/Compensation-Pfade,
- Human Approval als dauerhaften Workflow-Schritt.

Für solche Anforderungen bleibt eine zusätzliche Workflow-/Durability-Schicht sinnvoll.


---

# 14. OpenAI Dokumentation

Basis dieses Notebooks:

- Responses API Multi-agent  
  https://developers.openai.com/api/docs/guides/responses-multi-agent

Wichtige API-Elemente:

```python
client.beta.responses.create(
    ...,
    multi_agent={
        "enabled": True,
        "max_concurrent_subagents": 3,
    },
    betas=["responses_multi_agent=v1"],
)
```

Die Multi-agent-Item-Schemas sind Beta und können sich ändern.
